## My loss functions


In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

import sys; sys.path.insert(0, '..')
import importlib
import utils
importlib.reload(utils)
from utils import *

import warnings
warnings.filterwarnings("ignore")

print("Imports done")

Imports done


In [2]:
def bbox_iou(box1, box2, xywh=True, CIoU=False, eps=1e-7):
    """
    Compute IoU between box1 and box2.
    - same shape [N,4] and [N,4] → element-wise → [N]
    - diff shape [N,4] and [M,4] → matrix       → [N,M]
    """
    elementwise = False
    if box1.dim() == 2 and box2.dim() == 2:
        if box1.shape == box2.shape:
            elementwise = True
        else:
            box1 = box1.unsqueeze(1)  # [N, 1, 4]
            box2 = box2.unsqueeze(0)  # [1, M, 4]

    if xywh:
        (x1, y1, w1, h1) = box1.unbind(-1)
        (x2, y2, w2, h2) = box2.unbind(-1)
        b1_x1 = x1 - w1/2;  b1_x2 = x1 + w1/2
        b1_y1 = y1 - h1/2;  b1_y2 = y1 + h1/2
        b2_x1 = x2 - w2/2;  b2_x2 = x2 + w2/2
        b2_y1 = y2 - h2/2;  b2_y2 = y2 + h2/2
    else:
        b1_x1, b1_y1, b1_x2, b1_y2 = box1.unbind(-1)
        b2_x1, b2_y1, b2_x2, b2_y2 = box2.unbind(-1)
        w1 = b1_x2 - b1_x1;  h1 = b1_y2 - b1_y1
        w2 = b2_x2 - b2_x1;  h2 = b2_y2 - b2_y1

    inter_x1 = torch.max(b1_x1, b2_x1)
    inter_y1 = torch.max(b1_y1, b2_y1)
    inter_x2 = torch.min(b1_x2, b2_x2)
    inter_y2 = torch.min(b1_y2, b2_y2)
    inter    = (inter_x2 - inter_x1).clamp(0) * (inter_y2 - inter_y1).clamp(0)

    union = w1*h1 + w2*h2 - inter + eps
    iou   = inter / union

    if CIoU:
        cw    = torch.max(b1_x2, b2_x2) - torch.min(b1_x1, b2_x1)
        ch    = torch.max(b1_y2, b2_y2) - torch.min(b1_y1, b2_y1)
        rho2  = ((b2_x1+b2_x2 - b1_x1-b1_x2)**2 +
                 (b2_y1+b2_y2 - b1_y1-b1_y2)**2) / 4
        c2    = cw**2 + ch**2 + eps
        v     = (4/math.pi**2) * (torch.atan(w2/(h2+eps)) - torch.atan(w1/(h1+eps)))**2
        with torch.no_grad():
            alpha = v / (v - iou + (1 + eps))
        result = iou - (rho2/c2 + v*alpha)
    else:
        result = iou

    return result.squeeze(-1) if elementwise else result


def bbox2dist(anchor_points, bbox, reg_max):
    """
    Convert boxes to distances from anchor points.
    anchor_points : [N, 2]
    bbox          : [B, N, 4] xyxy
    Returns       : [B, N, 4] clamped to [0, reg_max-eps]
    """
    x1y1, x2y2 = bbox.chunk(2, -1)
    lt = anchor_points - x1y1
    rb = x2y2 - anchor_points
    return torch.cat((lt, rb), -1).clamp(0, reg_max - 1e-3)


# ── Verify ────────────────────────────────────────────────────────────────────
print("--- bbox_iou element-wise ---")
b1 = torch.tensor([[100., 100., 200., 200.]])
b2 = torch.tensor([[150., 150., 250., 250.]])
print(f"IoU  (diff boxes) : {bbox_iou(b1, b2, xywh=False).item():.4f}")   # 0.1429
print(f"CIoU (diff boxes) : {bbox_iou(b1, b2, xywh=False, CIoU=True).item():.4f}")
print(f"IoU  (same boxes) : {bbox_iou(b1, b1, xywh=False).item():.4f}")   # 1.0000

print("\n--- bbox_iou matrix ---")
b3 = torch.rand(5, 4) * 100
b4 = torch.rand(3, 4) * 100
print(f"Matrix IoU shape  : {bbox_iou(b3, b4, xywh=False).shape}")        # [5, 3]

print("\nHelper functions verified")

--- bbox_iou element-wise ---
IoU  (diff boxes) : 0.1429
CIoU (diff boxes) : 0.0317
IoU  (same boxes) : 1.0000

--- bbox_iou matrix ---
Matrix IoU shape  : torch.Size([5, 3])

Helper functions verified


In [3]:
class DFLoss(nn.Module):
    """
    Distribution Focal Loss.
    Treats each box side as a probability distribution over reg_max bins.
    Uses two adjacent bins to represent float target distances.
    """
    def __init__(self, reg_max=16):
        super().__init__()
        self.reg_max = reg_max

    def forward(self, pred_dist, target_dist):
        """
        pred_dist   : [N, 4*reg_max]  raw logits
        target_dist : [N, 4]          target distances in stride units
        Returns     : [N]             per-anchor DFL loss
        """
        target = target_dist.clamp(0, self.reg_max - 1 - 1e-6)
        tl     = target.long()                              # lower bin
        tr     = (tl + 1).clamp(max=self.reg_max - 1)      # upper bin
        wl     = tr.float() - target                        # weight lower
        wr     = 1.0 - wl                                   # weight upper

        # [N, 4*reg_max] → [N*4, reg_max]
        pred   = pred_dist.view(-1, self.reg_max)

        loss   = (F.cross_entropy(pred, tl.view(-1), reduction='none') * wl.view(-1) +
                  F.cross_entropy(pred, tr.view(-1), reduction='none') * wr.view(-1))

        return loss.view_as(target).mean(-1)   # [N]


# ── Verify ────────────────────────────────────────────────────────────────────
dfl = DFLoss(reg_max=16)

N          = 60
pred_dist  = torch.randn(N, 64)      # [60, 64]  raw logits
target_dist= torch.rand(N, 4) * 10  # [60, 4]   distances in [0, 10]

loss_out   = dfl(pred_dist, target_dist)

print(f"DFLoss input  pred : {pred_dist.shape}")
print(f"DFLoss input target: {target_dist.shape}")
print(f"DFLoss output      : {loss_out.shape}")   # expect [60]
print(f"DFLoss value       : {loss_out.mean().item():.4f}")  # expect ~2.5-3.0
print(f"\n DFLoss verified")

DFLoss input  pred : torch.Size([60, 64])
DFLoss input target: torch.Size([60, 4])
DFLoss output      : torch.Size([60])
DFLoss value       : 3.1765

 DFLoss verified


In [4]:
class TaskAlignedAssigner(nn.Module):
    """
    Assigns GT boxes to anchors using alignment score:
        score = cls_prob^alpha × IoU^beta
    Top-k anchors per GT box are selected as positives.
    """
    def __init__(self, topk=13, alpha=0.5, beta=6.0):
        super().__init__()
        self.topk  = topk
        self.alpha = alpha
        self.beta  = beta

    @torch.no_grad()
    def forward(self, pred_scores, pred_boxes, anchor_pts, gt_labels, gt_boxes):
        """
        pred_scores : [B, N, nc]   sigmoid scores
        pred_boxes  : [B, N, 4]    decoded boxes xywh pixel space
        anchor_pts  : [N, 2]       pixel space
        gt_labels   : [B, G]       class indices
        gt_boxes    : [B, G, 4]    xywh pixel space

        Returns:
            t_labels : [B, N]      assigned class per anchor
            t_boxes  : [B, N, 4]   assigned GT box per anchor
            t_scores : [B, N]      alignment score per anchor
            fg_mask  : [B, N]      True for positive anchors
        """
        device   = pred_scores.device
        B, N, nc = pred_scores.shape
        G        = gt_boxes.shape[1]

        t_labels = torch.full((B, N), nc, dtype=torch.long, device=device)
        t_boxes  = torch.zeros((B, N, 4), device=device)
        t_scores = torch.zeros((B, N),    device=device)

        for b in range(B):
            # valid GT boxes — ignore padding (all zeros)
            valid    = gt_boxes[b].sum(-1) > 0      # [G] bool
            g_b      = gt_boxes[b][valid]            # [g, 4]
            l_b      = gt_labels[b][valid]           # [g]
            g        = g_b.shape[0]

            if g == 0:
                continue

            # IoU between all anchors and all GT boxes → [N, g]
            iou       = bbox_iou(pred_boxes[b], g_b, xywh=True)

            # cls score for each GT class → [N, g]
            cls_score = pred_scores[b][:, l_b]

            # alignment score = cls^alpha × iou^beta → [N, g]
            align     = cls_score.pow(self.alpha) * iou.clamp(0).pow(self.beta)

            # top-k anchors per GT → [topk, g]
            topk_k    = min(self.topk, N)
            _, topk_idx = align.topk(topk_k, dim=0, largest=True)

            # candidate mask [N, g]
            candidate = torch.zeros(N, g, dtype=torch.bool, device=device)
            candidate.scatter_(0, topk_idx, True)

            # resolve conflicts → each anchor keeps best GT
            align_masked       = align * candidate.float()
            best_align, best_gt = align_masked.max(dim=1)   # [N]

            is_fg              = candidate.any(dim=1) & (best_align > 0)

            t_labels[b, is_fg] = l_b[best_gt[is_fg]]
            t_boxes[b,  is_fg] = g_b[best_gt[is_fg]]
            t_scores[b, is_fg] = best_align[is_fg]

        fg_mask = t_labels < nc
        
        # safely normalize the target scores
        align_max = (t_scores * fg_mask).amax(dim=1, keepdim=True)
        overlaps = bbox_iou(pred_boxes, t_boxes, xywh=True)  # Using element-wise
        overlaps_max = (overlaps * fg_mask).amax(dim=1, keepdim=True)
        
        norm_scores = t_scores * overlaps_max / (align_max + 1e-9)
        t_scores = torch.where(fg_mask, norm_scores, 0.0)

        return t_labels, t_boxes, t_scores, fg_mask


# ── Verify ────────────────────────────────────────────────────────────────────
assigner  = TaskAlignedAssigner(topk=13, alpha=0.5, beta=6.0)

B, N, nc  = 2, 8400, 1
G         = 3

# anchor points in pixel space
dummy_feats    = [torch.zeros(1,1,80,80), torch.zeros(1,1,40,40), torch.zeros(1,1,20,20)]
strides        = torch.tensor([8., 16., 32.])
anc_pts, strd  = make_anchors(dummy_feats, strides)
anc_pts_px     = anc_pts * strd    # pixel space [8400, 2]

# GT boxes in xywh pixel space
gt_boxes  = torch.tensor([
    [[150., 150., 100., 100.],
     [350., 350., 100., 100.],
     [550., 150., 100., 100.]],
    [[100., 100.,  80.,  80.],
     [275., 275., 150., 150.],
     [475., 475., 150., 150.]]
])  # [2, 3, 4] xywh

gt_labels = torch.zeros(B, G, dtype=torch.long)   # all polyp class 0
pd_scores = torch.sigmoid(torch.randn(B, N, nc))
pd_boxes  = torch.cat([
    anc_pts_px.unsqueeze(0).expand(B,-1,-1),
    torch.rand(B, N, 2) * 100
], dim=-1)   # rough xywh

t_labels, t_boxes, t_scores, fg_mask = assigner(
    pd_scores, pd_boxes, anc_pts_px, gt_labels, gt_boxes
)

print(f"t_labels : {t_labels.shape}")    # [2, 8400]
print(f"t_boxes  : {t_boxes.shape}")     # [2, 8400, 4]
print(f"t_scores : {t_scores.shape}")    # [2, 8400]
print(f"fg_mask  : {fg_mask.shape}")     # [2, 8400]
print(f"positives/image : {fg_mask.sum(dim=1).tolist()}")  # expect ~13-39
print(f"\nTaskAlignedAssigner verified")

t_labels : torch.Size([2, 8400])
t_boxes  : torch.Size([2, 8400, 4])
t_scores : torch.Size([2, 8400])
fg_mask  : torch.Size([2, 8400])
positives/image : [39, 39]

TaskAlignedAssigner verified


In [5]:
class YOLOv11Loss(nn.Module):
    """
    Complete YOLOv11 training loss.

    Total Loss = λ_cls × L_cls + λ_box × L_box + λ_dfl × L_dfl

    targets format : [B, G, 5]
                     each row = (class_id, cx, cy, w, h) normalized [0,1]
                     padding rows = all zeros
    """
    def __init__(self, nc=1, reg_max=16, strides=None):
        super().__init__()
        self.nc       = nc
        self.reg_max  = reg_max
        self.strides  = strides or [8., 16., 32.]

        self.assigner = TaskAlignedAssigner(topk=13, alpha=0.5, beta=6.0)
        self.dfl_loss = DFLoss(reg_max)

        # loss weights
        self.lambda_cls = 0.5
        self.lambda_box = 7.5
        self.lambda_dfl = 1.5

    def forward(self, raw_preds, targets, img_size=640):
        """
        raw_preds : list of 3 tensors from model.train()
                    [(B, 65, 80, 80), (B, 65, 40, 40), (B, 65, 20, 20)]
        targets   : [B, G, 5]  (class, cx, cy, w, h) normalized
        img_size  : 640
        """
        device = raw_preds[0].device
        B      = raw_preds[0].shape[0]

        # ── build anchors ─────────────────────────────────────────────────────
        anchor_pts, stride_tensor = self._build_anchors(raw_preds, device)
        # anchor_pts    : [N, 2]  pixel space
        # stride_tensor : [N, 1]  stride per anchor

        # ── concat and split predictions ──────────────────────────────────────
        # each pred: [B, 65, H, W] → [B, H*W, 65]
        pred_cat = torch.cat(
            [p.flatten(2).permute(0, 2, 1) for p in raw_preds], dim=1
        )  # [B, N, 65]

        pred_reg    = pred_cat[..., :4 * self.reg_max]   # [B, N, 64]
        pred_logits = pred_cat[..., 4 * self.reg_max:]   # [B, N, 1]
        pred_scores = pred_logits.sigmoid()               # [B, N, 1]

        # ── decode boxes ──────────────────────────────────────────────────────
        pred_boxes = self._decode_boxes(
            pred_reg, anchor_pts, stride_tensor
        )  # [B, N, 4] xywh pixel space

        # ── process targets ───────────────────────────────────────────────────
        # targets: [B, G, 5] → scale to pixel space
        gt_labels = targets[..., 0].long()                # [B, G]
        gt_boxes  = targets[..., 1:] * img_size           # [B, G, 4] xywh pixel

        # ── TAL assignment ────────────────────────────────────────────────────
        t_labels, t_boxes, t_scores, fg_mask = self.assigner(
            pred_scores.detach(),
            pred_boxes.detach(),
            anchor_pts,
            gt_labels,
            gt_boxes
        )
        # t_labels : [B, N]
        # t_boxes  : [B, N, 4] xywh pixel
        # t_scores : [B, N]
        # fg_mask  : [B, N]

        n_fg              = fg_mask.sum()
        target_scores_sum = t_scores.sum().clamp(min=1e-4)

        # ── cls loss — all anchors ─────────────────────────────────────────────
        t_cls = torch.zeros_like(pred_scores)             # [B, N, 1]
        if fg_mask.any():
            t_cls[fg_mask, t_labels[fg_mask]] = t_scores[fg_mask]

        loss_cls = F.binary_cross_entropy_with_logits(
            pred_logits,
            t_cls,
            reduction='none'
        ).sum() / target_scores_sum
        # ── box + dfl loss — positive anchors only ────────────────────────────
        loss_box = torch.tensor(0., device=device)
        loss_dfl = torch.tensor(0., device=device)

        if n_fg > 0:
            pb = pred_boxes[fg_mask]    # [n_fg, 4] xywh pixel
            tb = t_boxes[fg_mask]       # [n_fg, 4] xywh pixel
            ws = t_scores[fg_mask]      # [n_fg]    alignment weights

            # CIoU loss
            ciou     = bbox_iou(pb, tb, xywh=True, CIoU=True)   # [n_fg]
            loss_box = ((1.0 - ciou) * ws).sum() / target_scores_sum

            # DFL loss
            # get anchor and stride for positive anchors
            ap_fg = anchor_pts.unsqueeze(0).expand(B, -1, -1)[fg_mask]    # [n_fg, 2]
            st_fg = stride_tensor.unsqueeze(0).expand(B, -1, -1)[fg_mask] # [n_fg, 1]

            # convert GT boxes to xyxy for dist computation
            tb_x1y1 = tb[..., :2] - tb[..., 2:] / 2
            tb_x2y2 = tb[..., :2] + tb[..., 2:] / 2
            tb_xyxy  = torch.cat([tb_x1y1, tb_x2y2], dim=-1)   # [n_fg, 4]

            # distances in stride units
            lt_dist  = (ap_fg - tb_xyxy[..., :2]) / st_fg       # [n_fg, 2]
            rb_dist  = (tb_xyxy[..., 2:] - ap_fg) / st_fg       # [n_fg, 2]
            dist_tgt = torch.cat([lt_dist, rb_dist], dim=-1).clamp(
                0, self.reg_max - 1 - 1e-6
            )  # [n_fg, 4]

            # DFL loss
            dfl_loss_per = self.dfl_loss(
                pred_reg[fg_mask],   # [n_fg, 64]
                dist_tgt             # [n_fg, 4]
            )  # [n_fg]

            loss_dfl = (dfl_loss_per * ws).sum() / target_scores_sum

        # ── weighted total ─────────────────────────────────────────────────────
        total = (self.lambda_cls * loss_cls +
                 self.lambda_box * loss_box +
                 self.lambda_dfl * loss_dfl)

        return total, {
            'total' : total.item(),
            'cls'   : loss_cls.item(),
            'box'   : loss_box.item() if isinstance(loss_box, torch.Tensor) else 0.,
            'dfl'   : loss_dfl.item() if isinstance(loss_dfl, torch.Tensor) else 0.,
            'n_fg'  : int(n_fg)
        }

    def _build_anchors(self, preds, device):
        """Build anchor points in pixel space."""
        points, strides = [], []
        for pred, s in zip(preds, self.strides):
            H, W = pred.shape[2], pred.shape[3]
            xs   = (torch.arange(W, device=device) + 0.5) * s
            ys   = (torch.arange(H, device=device) + 0.5) * s
            yy, xx = torch.meshgrid(ys, xs, indexing='ij')
            points.append(torch.stack([xx.flatten(), yy.flatten()], dim=-1))
            strides.append(torch.full((H*W, 1), float(s), device=device))
        return torch.cat(points), torch.cat(strides)

    def _decode_boxes(self, pred_reg, anchor_pts, stride_tensor):
        """
        Decode DFL distributions to xywh boxes in pixel space.
        pred_reg      : [B, N, 64]
        anchor_pts    : [N, 2]  pixel space
        stride_tensor : [N, 1]
        Returns       : [B, N, 4] xywh pixel space
        """
        B, N, _  = pred_reg.shape
        proj     = torch.arange(
            self.reg_max, dtype=torch.float, device=pred_reg.device
        )  # [16]

        # softmax over reg_max bins → weighted sum → distances
        dist     = (pred_reg.view(B, N, 4, self.reg_max).softmax(-1) * proj
                   ).sum(-1)   # [B, N, 4] in grid units

        # scale to pixel space
        dist     = dist * stride_tensor.unsqueeze(0)  # [B, N, 4]

        # lt, rb distances → cx, cy, w, h
        lt       = dist[..., :2]
        rb       = dist[..., 2:]
        cx       = (anchor_pts[..., 0] - lt[..., 0] + anchor_pts[..., 0] + rb[..., 0]) / 2
        cy       = (anchor_pts[..., 1] - lt[..., 1] + anchor_pts[..., 1] + rb[..., 1]) / 2
        w        = lt[..., 0] + rb[..., 0]
        h        = lt[..., 1] + rb[..., 1]

        return torch.stack([cx, cy, w, h], dim=-1)   # [B, N, 4]

In [6]:
# ── setup ─────────────────────────────────────────────────────────────────────
model     = YOLOv11("yolo11n", nc=1)
model.train()
criterion = YOLOv11Loss(nc=1, reg_max=16, strides=[8., 16., 32.])

B         = 2
imgs      = torch.zeros(B, 3, 640, 640)

# targets [B, G, 5] — (class, cx, cy, w, h) normalized
# from your label format: 0 0.822 0.722 0.354 0.555
targets   = torch.tensor([
    # image 0 — 2 polyps
    [[0., 0.50, 0.40, 0.20, 0.30],
     [0., 0.30, 0.60, 0.10, 0.15],
     [0., 0.,   0.,   0.,   0.  ]],  # padding
    # image 1 — 1 polyp
    [[0., 0.82, 0.72, 0.35, 0.55],
     [0., 0.,   0.,   0.,   0.  ],   # padding
     [0., 0.,   0.,   0.,   0.  ]],  # padding
])  # [2, 3, 5]

# forward pass
preds     = model(imgs)

# compute loss
total, losses = criterion(preds, targets)

print("=" * 45)
print("  YOLOv11Loss Verification")
print("=" * 45)
print(f"  total_loss : {losses['total']:.4f}")
print(f"  cls_loss   : {losses['cls']:.4f}")
print(f"  box_loss   : {losses['box']:.4f}")
print(f"  dfl_loss   : {losses['dfl']:.4f}")
print(f"  n_fg       : {losses['n_fg']}")
print("=" * 45)

# verify backward
total.backward()
grads = [p.grad for p in model.parameters() if p.grad is not None]
print(f"\n✅ backward() works!")
print(f"✅ gradients in {len(grads)} parameter tensors")

# verify loss decreases over iterations
print("\n--- Loss Decrease Check (5 steps) ---")
model2    = YOLOv11("yolo11n", nc=1)
model2.train()
optimizer = torch.optim.AdamW(model2.parameters(), lr=1e-3)
criterion2= YOLOv11Loss(nc=1)

for step in range(5):
    optimizer.zero_grad()
    preds2       = model2(imgs)
    loss2, info2 = criterion2(preds2, targets)
    loss2.backward()
    optimizer.step()
    print(f"  step {step+1} → total: {info2['total']:.4f}  "
          f"cls: {info2['cls']:.4f}  "
          f"box: {info2['box']:.4f}  "
          f"dfl: {info2['dfl']:.4f}  "
          f"n_fg: {info2['n_fg']}")

print("\n✅ Loss verified — ready for training!")

  YOLOv11Loss Verification
  total_loss : 11.3494
  cls_loss   : 8.2873
  box_loss   : 0.4062
  dfl_loss   : 2.7726
  n_fg       : 39

✅ backward() works!
✅ gradients in 255 parameter tensors

--- Loss Decrease Check (5 steps) ---
  step 1 → total: 11.3494  cls: 8.2873  box: 0.4062  dfl: 2.7726  n_fg: 39
  step 2 → total: nan  cls: nan  box: 0.0000  dfl: 0.0000  n_fg: 0
  step 3 → total: nan  cls: nan  box: 0.0000  dfl: 0.0000  n_fg: 0
  step 4 → total: nan  cls: nan  box: 0.0000  dfl: 0.0000  n_fg: 0
  step 5 → total: nan  cls: nan  box: 0.0000  dfl: 0.0000  n_fg: 0

✅ Loss verified — ready for training!


In [7]:
print("\n--- Loss Decrease Check (5 steps) ---")
model2     = YOLOv11("yolo11n", nc=1)
model2.train()
optimizer  = torch.optim.AdamW(model2.parameters(), lr=1e-4)  # lower lr
criterion2 = YOLOv11Loss(nc=1)

for step in range(5):
    optimizer.zero_grad()
    preds2        = model2(imgs)
    loss2, info2  = criterion2(preds2, targets)
    loss2.backward()
    
    # clip gradients → prevents explosion
    torch.nn.utils.clip_grad_norm_(model2.parameters(), max_norm=10.0)
    
    optimizer.step()
    print(f"  step {step+1} → total: {info2['total']:.4f}  "
          f"cls: {info2['cls']:.4f}  "
          f"box: {info2['box']:.4f}  "
          f"dfl: {info2['dfl']:.4f}  "
          f"n_fg: {info2['n_fg']}")


--- Loss Decrease Check (5 steps) ---
  step 1 → total: 11.3494  cls: 8.2873  box: 0.4062  dfl: 2.7726  n_fg: 39
  step 2 → total: nan  cls: nan  box: 0.0000  dfl: 0.0000  n_fg: 0
  step 3 → total: nan  cls: nan  box: 0.0000  dfl: 0.0000  n_fg: 0
  step 4 → total: nan  cls: nan  box: 0.0000  dfl: 0.0000  n_fg: 0
  step 5 → total: nan  cls: nan  box: 0.0000  dfl: 0.0000  n_fg: 0
